In [1]:
import warnings
import os
import requests
warnings.filterwarnings('ignore')

In [2]:
!pip install -qU langchain-community bs4


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [56]:
import re
from langchain_core.documents import Document
from langchain_community.document_loaders import BSHTMLLoader

loader = BSHTMLLoader("./rsssf_worldcup/miscellaneous/wcwinners.html")
docs = loader.load()
text = docs[0].page_content

pattern = r'(?=^\d{4}\s+.+$)'
sections = re.split(pattern, text, flags=re.MULTILINE)
sections = [s.strip() for s in sections if s.strip()]

documents = []
for section in sections:
    first_line = section.splitlines()[0].strip()
    documents.append(
        Document(
            page_content=section,
            metadata={
                "source": "wcwinners.html",
                "section_title": first_line,
            }
        )
    )

In [57]:
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_openai import ChatOpenAI
from IPython.display import display, Markdown

In [58]:
embedding = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = DocArrayInMemorySearch.from_documents(
    documents,   # или chunks
    embedding=embedding
)


In [121]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

In [147]:
query = "How many times did Brazil win the World Cup?"

results = retriever.get_relevant_documents(query)

for i, doc in enumerate(results, 1):
    print(f"--- result {i} ---")
    print(doc.metadata)
    print(doc.page_content[:100])

--- result 1 ---
{'source': 'wcwinners.html', 'section_title': 'World Cup Champions Squads 1930 - 2022'}
World Cup Champions Squads 1930 - 2022


World Cup Champions Squads 1930 - 2022




This file lists 
--- result 2 ---
{'source': 'wcwinners.html', 'section_title': '2014 Germany'}
2014 Germany

Nr.     Name                      Team             Born            M Pl.  Mins   Goals
--- result 3 ---
{'source': 'wcwinners.html', 'section_title': '1998 France'}
1998 France

16       Fabien BARTHEZ        28.06.71 183/76  AS Monaco             7 (684)- 2
 5    
--- result 4 ---
{'source': 'wcwinners.html', 'section_title': '2018 France'}
2018 France

Nr.     Name                   Team                Born            M Pl.  Mins   Goals 
--- result 5 ---
{'source': 'wcwinners.html', 'section_title': '1974 West Germany'}
1974 West Germany

 5          Franz BECKENBAUER   11.09.45 181/75  Bayern München          7 (630)

--- result 6 ---
{'source': 'wcwinners.html', 'section_title': '1970 Br

In [148]:
context = "\n\n".join(doc.page_content for doc in results)

prompt = f"""
Answer only from the provided context.
If the answer is not in the context, say you don't know.

Question: {query}

Context:
{context}
"""

In [149]:
# llm_model = OllamaLLM(
#     model="llama3",
#     temperature=0,
# )

from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("DEEPSEEK_API_KEY")

llm_model = ChatOpenAI(
    api_key=api_key,
    model="deepseek-chat",
    temperature=0,
    base_url="https://api.deepseek.com"
)

In [150]:
response = llm_model.call_as_llm(prompt)

In [151]:
display(Markdown(response))

I don't know.